# SL Chess Bot — train a strong-enough-to-beat-friends bot via supervised Stockfish labels

**Goal**: train a ~40M-param ChessTransformer on ~700-900k Stockfish-labeled positions from Lichess Elite (Sep+Oct+Nov 2025) — Ruoss et al. 2024 territory at smaller scale. Target Elo 2400-2700.

**Pipeline**:

1. GPU + Drive
2. Clone + install
3. wandb headless auth
4. Download Lichess Elite (handled locally; just verify .npz on Drive)
5. Train SL bot ~30-40h on G4 — uses the 50h Colab budget close to fully
6. Eval ladder vs Stockfish skill {0, 3, 5, 7, 10, 15} with 200 games per skill (~30 min)
7. Save final ckpt to Drive
8. Play instructions

**Recommended GPU**: G4 (RTX PRO 6000 Blackwell, 96 GB).

## 1. GPU + Drive

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_BASE = '/content/drive/MyDrive/sampling-chess'
os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/sl_bot', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/sl_bot/data', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/sl_bot/checkpoints', exist_ok=True)
print('Drive base:', DRIVE_BASE)

## 2. Clone + install

In [ ]:
%cd /content
![ -d sampling-chess ] || git clone https://github.com/hongttisme/sampling-chess.git
%cd sampling-chess
!git pull --ff-only

In [ ]:
# SL bot only needs the lighter ml stack (no pgx required for ChessTransformer + chess.Board pipeline).
!pip install -q -e ".[ml]"
!apt-get install -qq -y stockfish && which stockfish

In [ ]:
import jax, flax, optax
print(f'jax {jax.__version__} | flax {flax.__version__} | optax {optax.__version__}')
print(f'devices: {jax.devices()} | backend: {jax.default_backend()}')

## 3. wandb headless auth

One-time: `!echo "YOUR_API_KEY" > /content/drive/MyDrive/sampling-chess/wandb_key.txt`.

In [ ]:
import os
KEY_PATH = '/content/drive/MyDrive/sampling-chess/wandb_key.txt'
if os.path.exists(KEY_PATH):
    with open(KEY_PATH) as f:
        os.environ['WANDB_API_KEY'] = f.read().strip()
    print(f'wandb: loaded API key from {KEY_PATH}')
else:
    print(f'wandb: no key file at {KEY_PATH}; the next cell will block on input')
!wandb login

## 4. Download Lichess Elite Database

~700MB-2GB compressed PGN of 2300+ rated Lichess games. To use a different month, pass `--url` explicitly. The downloader unzips into `data/` so the labeler can stream the .pgn directly.

In [ ]:
!python scripts/10_download_data.py --out-dir /content/drive/MyDrive/sampling-chess/sl_bot/data
!ls -lah /content/drive/MyDrive/sampling-chess/sl_bot/data/ | head

## 5. Label positions with Stockfish

**Recommended**: do this on your LOCAL 12-core machine overnight, then upload `labels_elite_500k.npz` to Drive. Local cmd:

```bash
.venv/bin/python scripts/02_label_batch.py --source pgn \
    --pgn-path data/lichess_elite_2025-11.pgn --n 500000 \
    --out data/labels_elite_500k.npz \
    --workers 12 --depth 12 --multipv 5
```

Then upload the `.npz` to `/content/drive/MyDrive/sampling-chess/sl_bot/data/labels_elite_500k.npz`.

**Fallback (Colab CPU)**: uncomment + run the cell below. Slow (~20h on Colab's shared CPU) — burns Colab session time but no local setup needed. Keep `--n` modest (e.g., 100k) if going this route.

In [ ]:
# Skip if labels already on Drive (uploaded from local labeling).
import os
LABEL_PATH = '/content/drive/MyDrive/sampling-chess/sl_bot/data/labels_elite_500k.npz'
if os.path.exists(LABEL_PATH):
    import numpy as np
    d = np.load(LABEL_PATH)
    print(f'[ok] {LABEL_PATH}')
    print(f'  fens         : {d["fens"].shape}')
    print(f'  move_indices : {d["move_indices"].shape}')
    print(f'  value_targets: {d["value_targets"].shape}')
else:
    print(f'[need labels] {LABEL_PATH} not present')
    print('  Either upload from local labeling, OR run the next cell to label on Colab CPU.')

In [ ]:
# Colab-CPU labeling fallback. Slow. Uncomment to use.
# Find PGN under data/ and label N positions; reduce --n if budget tight.
# !PGN=$(ls /content/drive/MyDrive/sampling-chess/sl_bot/data/*.pgn 2>/dev/null | head -1) && \
#     echo "PGN=$PGN" && \
#     python scripts/02_label_batch.py --source pgn \
#         --pgn-path "$PGN" --n 100000 \
#         --out /content/drive/MyDrive/sampling-chess/sl_bot/data/labels_elite_100k.npz \
#         --workers 4 --depth 12 --multipv 5

In [ ]:
# Train ~40M-param ChessTransformer (12 layers, d_model=512, n_heads=8, ffn_dim=2048)
# 200k steps x batch 1024 with AdamW lr 3e-4 + cosine decay + 1k warmup.
# Wandb-logged. Ckpt every 5k steps to Drive (resume-safe via --no-resume opt-out).
#
# Wall-clock estimate on G4: ~30-40h (uses most of the 50h Colab budget).
# A Colab disconnect at any point is recoverable: re-run this cell and it
# auto-resumes from latest.pkl.

!python scripts/03_sl_train.py \
    --data /content/drive/MyDrive/sampling-chess/sl_bot/data/labels_elite_combined.npz \
    --steps 200000 --batch-size 1024 --lr 3e-4 --warmup 1000 --lambda-v 1.0 \
    --n-layers 12 --d-model 512 --n-heads 8 --ffn-dim 2048 \
    --log-every 100 --ckpt-every 5000 \
    --ckpt-dir /content/drive/MyDrive/sampling-chess/sl_bot/checkpoints \
    --wandb --name sl-bot-40M-200k --group sl-bot \
    2>&1 | tee /content/drive/MyDrive/sampling-chess/sl_bot/train.log

In [ ]:
!python scripts/03_sl_train.py \
    --data /content/drive/MyDrive/sampling-chess/sl_bot/data/labels_elite_500k.npz \
    --steps 100000 --batch-size 1024 --lr 3e-4 --warmup 1000 --lambda-v 1.0 \
    --log-every 100 --ckpt-every 5000 \
    --ckpt-dir /content/drive/MyDrive/sampling-chess/sl_bot/checkpoints \
    --wandb --name sl-bot-elite500k-100k --group sl-bot \
    2>&1 | tee /content/drive/MyDrive/sampling-chess/sl_bot/train.log

In [ ]:
# Eval ladder: 200 games per skill across {0, 3, 5, 7, 10, 15}.
# Skill 0 ~1300 Elo, 5 ~1900, 10 ~2200, 15 ~2500.
import pickle, jax, jax.numpy as jnp, numpy as np, chess, glob
from sampling_chess import board as B, eval as E
from sampling_chess.net import ChessTransformer, count_params

ckpt_path = '/content/drive/MyDrive/sampling-chess/sl_bot/checkpoints/latest.pkl'
print(f'[eval] using {ckpt_path}')
with open(ckpt_path, 'rb') as f:
    ckpt = pickle.load(f)
params = ckpt['params']
config = ckpt.get('config', {}) or {}
model = ChessTransformer(
    n_layers=int(config.get('n_layers', 12)),
    d_model=int(config.get('d_model', 512)),
    n_heads=int(config.get('n_heads', 8)),
    ffn_dim=int(config.get('ffn_dim', 2048)),
)
print(f'[model] {count_params(params):,} params')

@jax.jit
def fwd(params, pieces, globals_, mask):
    logits, value = model.apply({'params': params}, pieces[None].astype(jnp.int32), globals_[None])
    return jnp.where(mask, logits[0], jnp.float32(-1e9))

def policy_fn(board: chess.Board) -> chess.Move:
    pieces = jnp.asarray(B.board_to_planes(board))
    globals_ = jnp.asarray(B.board_to_global(board))
    mask = jnp.asarray(B.legal_action_mask(board))
    masked = fwd(params, pieces, globals_, mask)
    return B.index_to_move(int(jnp.argmax(masked)))

results = {}
for skill in [0, 3, 5, 7, 10, 15]:
    print(f'\n--- vs Stockfish skill {skill} ---')
    result = E.play_match(
        policy_fn, opponent_skill=skill, n_games=200,
        opponent_time=0.05, max_plies=300, seed=skill,
    )
    results[skill] = result
    print(f'  {result.summary()}')

# Print Elo curve summary table.
print('\n=== Elo summary ===')
print(f'{"skill":>6}  {"score":>7}  {"elo":>7}  {"95% CI Elo":>20}')
for skill, r in results.items():
    lo, hi = r.elo_ci()
    print(f'{skill:>6}  {r.score:>7.3f}  {r.elo():>+7.0f}  [{lo:>+7.0f}, {hi:>+7.0f}]')

In [ ]:
# In-line eval driver (uses make_random_policy / play_match from sampling_chess.eval).
import pickle, jax, jax.numpy as jnp, numpy as np, chess, glob
from sampling_chess import board as B, eval as E
from sampling_chess.net import ChessTransformer, count_params

ckpts = sorted(glob.glob('/content/drive/MyDrive/sampling-chess/sl_bot/checkpoints/ckpt_*.pkl'))
assert ckpts, 'no checkpoints found'
ckpt_path = ckpts[-1]
print(f'[eval] using {ckpt_path}')
with open(ckpt_path, 'rb') as f:
    ckpt = pickle.load(f)
params = ckpt['params']
config = ckpt.get('config', {}) or {}
model = ChessTransformer(
    n_layers=int(config.get('n_layers', 8)),
    d_model=int(config.get('d_model', 384)),
    n_heads=int(config.get('n_heads', 6)),
    ffn_dim=int(config.get('ffn_dim', 1536)),
)
print(f'[model] {count_params(params):,} params')

@jax.jit
def fwd(params, pieces, globals_, mask):
    logits, value = model.apply({'params': params}, pieces[None].astype(jnp.int32), globals_[None])
    return jnp.where(mask, logits[0], jnp.float32(-1e9))

def policy_fn(board: chess.Board) -> chess.Move:
    pieces = jnp.asarray(B.board_to_planes(board))
    globals_ = jnp.asarray(B.board_to_global(board))
    mask = jnp.asarray(B.legal_action_mask(board))
    masked = fwd(params, pieces, globals_, mask)
    return B.index_to_move(int(jnp.argmax(masked)))

for skill in [0, 3, 5, 7, 10]:
    print(f'\n--- vs Stockfish skill {skill} ---')
    result = E.play_match(
        policy_fn, opponent_skill=skill, n_games=100,
        opponent_time=0.05, max_plies=300, seed=skill,
    )
    print(f'  {result.summary()}')

## 8. Notes — how to play

1. Pick the latest `ckpt_*.pkl` from `/content/drive/MyDrive/sampling-chess/sl_bot/checkpoints/` — that's your final bot.
2. Download it locally (or run from Colab via `!python scripts/11_play_vs_human.py ...`).
3. Play:

```bash
python scripts/11_play_vs_human.py --ckpt path/to/ckpt_0100000.pkl
# play as black:
python scripts/11_play_vs_human.py --ckpt ... --human-color black
# more variety in bot moves:
python scripts/11_play_vs_human.py --ckpt ... --mode sample --temperature 0.5
```

Type UCI moves at the prompt (e.g., `e2e4`, `e7e8q` for queen-promo, `e1g1` for white kingside castle). `quit` / `q` ends the game.

## 9. Send to friend

- Share `scripts/11_play_vs_human.py` + the final ckpt + a `pip install chess jax flax` instruction. Or:
- Wrap it in a Colab notebook for friend to play in the browser.
- Or build a lichess.org bot account and stream moves via lichess-bot (more work; consider after the local play UI works).